In [212]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,classification_report,mean_absolute_error
import requests
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler

In [213]:

API_KEY = "RXRKOPYRVWQ4V2SZ"

symbol = "IBM"

url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}"

response = requests.get(url)

data = response.json()

print(data)

{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'IBM', '3. Last Refreshed': '2026-06-08', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}, 'Time Series (Daily)': {'2026-06-08': {'1. open': '286.4400', '2. high': '290.5000', '3. low': '279.4300', '4. close': '280.8200', '5. volume': '6790639'}, '2026-06-05': {'1. open': '300.0000', '2. high': '302.3000', '3. low': '281.0701', '4. close': '284.8400', '5. volume': '12509480'}, '2026-06-04': {'1. open': '307.4250', '2. high': '310.4400', '3. low': '300.1800', '4. close': '301.7700', '5. volume': '9608097'}, '2026-06-03': {'1. open': '318.2950', '2. high': '318.2950', '3. low': '302.5301', '4. close': '305.6300', '5. volume': '13926558'}, '2026-06-02': {'1. open': '313.7500', '2. high': '332.4600', '3. low': '310.1090', '4. close': '329.2300', '5. volume': '17210696'}, '2026-06-01': {'1. open': '322.5500', '2. high': '327.9800', '3. low': '308.0000', '4. close': '320.4200', '5. v

In [214]:
time_series = data["Time Series (Daily)"]

df = pd.DataFrame(time_series).T

print(df.head())

             1. open   2. high    3. low  4. close 5. volume
2026-06-08  286.4400  290.5000  279.4300  280.8200   6790639
2026-06-05  300.0000  302.3000  281.0701  284.8400  12509480
2026-06-04  307.4250  310.4400  300.1800  301.7700   9608097
2026-06-03  318.2950  318.2950  302.5301  305.6300  13926558
2026-06-02  313.7500  332.4600  310.1090  329.2300  17210696


In [215]:
df

,1. open,2. high,3. low,4. close,5. volume
2026-06-08,286.4400,290.5000,279.4300,280.8200,6790639
2026-06-05,300.0000,302.3000,281.0701,284.8400,12509480
2026-06-04,307.4250,310.4400,300.1800,301.7700,9608097
2026-06-03,318.2950,318.2950,302.5301,305.6300,13926558
2026-06-02,313.7500,332.4600,310.1090,329.2300,17210696
...,...,...,...,...,...
2026-01-21,292.7600,297.6700,292.5100,297.5400,5185023
2026-01-20,301.3500,301.6000,290.1600,291.3500,7211706
2026-01-16,301.0000,307.4500,300.7800,305.6700,6199635
2026-01-15,309.0000,311.8800,297.0400,297.9500,4932480


In [216]:
df.dtypes

1. open      str
2. high      str
3. low       str
4. close     str
5. volume    str
dtype: object

In [217]:
df = df.astype(float)

print(df.dtypes)

1. open      float64
2. high      float64
3. low       float64
4. close     float64
5. volume    float64
dtype: object


In [218]:
df.index = pd.to_datetime(df.index)

In [219]:
df = df.sort_index()

In [220]:
df

,1. open,2. high,3. low,4. close,5. volume
2026-01-14,303.500,309.190,301.5000,309.03,3779045.0
2026-01-15,309.000,311.880,297.0400,297.95,4932480.0
2026-01-16,301.000,307.450,300.7800,305.67,6199635.0
2026-01-20,301.350,301.600,290.1600,291.35,7211706.0
2026-01-21,292.760,297.670,292.5100,297.54,5185023.0
...,...,...,...,...,...
2026-06-02,313.750,332.460,310.1090,329.23,17210696.0
2026-06-03,318.295,318.295,302.5301,305.63,13926558.0
2026-06-04,307.425,310.440,300.1800,301.77,9608097.0
2026-06-05,300.000,302.300,281.0701,284.84,12509480.0


In [221]:
df.columns = ["Open", "High", "Low", "Close", "Volume"]

In [222]:
df["Target"] = df["Close"].shift(-1)

In [223]:
df

,Open,High,Low,Close,Volume,Target
2026-01-14,303.500,309.190,301.5000,309.03,3779045.0,297.95
2026-01-15,309.000,311.880,297.0400,297.95,4932480.0,305.67
2026-01-16,301.000,307.450,300.7800,305.67,6199635.0,291.35
2026-01-20,301.350,301.600,290.1600,291.35,7211706.0,297.54
2026-01-21,292.760,297.670,292.5100,297.54,5185023.0,294.67
...,...,...,...,...,...,...
2026-06-02,313.750,332.460,310.1090,329.23,17210696.0,305.63
2026-06-03,318.295,318.295,302.5301,305.63,13926558.0,301.77
2026-06-04,307.425,310.440,300.1800,301.77,9608097.0,284.84
2026-06-05,300.000,302.300,281.0701,284.84,12509480.0,280.82


In [224]:
df.isnull().sum()

Open      0
High      0
Low       0
Close     0
Volume    0
Target    1
dtype: int64

In [225]:
df.dropna(inplace=True)

In [226]:
df.isnull().sum()

Open      0
High      0
Low       0
Close     0
Volume    0
Target    0
dtype: int64

In [227]:
x=df.drop("Target",axis=1)
y=df["Target"]

print(x.head())
print(y.head())

              Open    High     Low   Close     Volume
2026-01-14  303.50  309.19  301.50  309.03  3779045.0
2026-01-15  309.00  311.88  297.04  297.95  4932480.0
2026-01-16  301.00  307.45  300.78  305.67  6199635.0
2026-01-20  301.35  301.60  290.16  291.35  7211706.0
2026-01-21  292.76  297.67  292.51  297.54  5185023.0
2026-01-14    297.95
2026-01-15    305.67
2026-01-16    291.35
2026-01-20    297.54
2026-01-21    294.67
Name: Target, dtype: float64


In [228]:
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,shuffle=False
)

In [229]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [230]:
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

In [231]:
model.fit(x_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [232]:
y_pred = model.predict(x_test)

In [233]:
mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)

print("R2 Score:", r2)

Mean Absolute Error: 15.540059722900395
R2 Score: 0.7006092544753686


In [234]:
result = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

print(result.head())

            Actual   Predicted
2026-05-08  223.55  230.915955
2026-05-11  219.22  231.760834
2026-05-12  214.64  232.239990
2026-05-13  218.37  232.401108
2026-05-14  219.30  232.239990
